# `test_model_components.py` — Unit Tests for Model Architecture

## Purpose
Unit-style tests for every model component. **No trained checkpoint required** — all tests
use random inputs and only verify output *shapes* and that no exception is raised.

Run order within the test suite:
| Test | What it checks |
|------|----------------|
| 1. `AspectAwareRoBERTa` | Forward-pass shapes for all 4 flag combinations |
| 2. `MultiAspectSentimentModel` | GCN on/off + return_attention mode |
| 3. Loss functions | FocalLoss, HybridLoss, AspectSpecificLossManager |
| 4. Ablation modes | A1/A2/A5/combined configs all forward-pass cleanly |
| 5. Parameter count | Total trainable params > 50 M (sanity check) |

**Run from:** `ml-research/` directory (or any directory — paths are resolved automatically)

In [ ]:
import sys, os, torch

# ── Resolve paths so this notebook works from any directory ──────────────────
NOTEBOOK_DIR = os.path.abspath('')
# Walk up until we find 'tests/' as a child — that is the ml-research root
PROJECT_ROOT = NOTEBOOK_DIR
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT:  # reached filesystem root
        break
    PROJECT_ROOT = parent

SRC_DIR   = os.path.join(PROJECT_ROOT, 'src')
UTILS_DIR = os.path.join(PROJECT_ROOT, 'utils')

for p in [PROJECT_ROOT, SRC_DIR, UTILS_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Project root:', PROJECT_ROOT)

# ── Import model and loss components ────────────────────────────────────────
import yaml
from models.model  import AspectAwareRoBERTa, MultiAspectSentimentModel, create_model
from models.losses import FocalLoss, HybridLoss, AspectSpecificLossManager

# ── Helper printers ──────────────────────────────────────────────────────────
def sep(title=''):  print(f"\n{'='*60}\n  {title}\n{'='*60}" if title else f"\n{'='*60}")
def ok(msg):   print(f'  [OK ] {msg}')
def err(msg):  print(f'  [ERR] {msg}')

print('[OK] Imports successful')

In [ ]:
# ── Load config.yaml ────────────────────────────────────────────────────────
cfg_path = os.path.join(PROJECT_ROOT, 'configs', 'config.yaml')
with open(cfg_path) as f:
    config = yaml.safe_load(f)
ok(f'Config loaded from {cfg_path}')
print(f'  Model: {config["model"]["roberta_model"]}  |  '
      f'aspects: {config["model"]["num_aspects"]}  |  '
      f'classes: {config["model"]["num_classes"]}')

## Test 1 — `AspectAwareRoBERTa` Forward Pass

Tests all 4 combinations of the two boolean flags:
- `use_aspect_attention` (True/False) — uses aspect embedding as MHA query vs. plain CLS
- `use_shared_classifier` (True/False) — one shared MLP vs. per-aspect MLPs

**Expected output shapes** for batch size 2, sequence length 64:
- `preds`: `(2, num_classes)`
- `attn_weights`: `(2, seq_len)`
- `aspect_repr`: `(2, hidden_dim)`

In [ ]:
sep('Test 1: AspectAwareRoBERTa — all 4 flag combinations')
B, S = 2, 64
model_cfg = config['model']

for use_attn in [True, False]:
    for use_shared in [True, False]:
        label = f'attention={use_attn}, shared_clf={use_shared}'
        try:
            m = AspectAwareRoBERTa(
                roberta_model=model_cfg['roberta_model'],
                num_aspects=model_cfg['num_aspects'],
                num_classes=model_cfg['num_classes'],
                hidden_dim=model_cfg['hidden_dim'],
                dropout=0.0,
                use_aspect_attention=use_attn,
                use_shared_classifier=use_shared,
            )
            m.eval()
            ids  = torch.randint(0, 1000, (B, S))
            mask = torch.ones(B, S, dtype=torch.long)
            asp  = torch.zeros(B, dtype=torch.long)
            with torch.no_grad():
                preds, attn_w, asp_repr = m(ids, mask, asp)
            assert preds.shape    == (B, model_cfg['num_classes'])
            assert attn_w.shape   == (B, S)
            assert asp_repr.shape == (B, model_cfg['hidden_dim'])
            ok(label)
        except Exception as exc:
            err(f'{label}: {exc}')

## Test 2 — Full `MultiAspectSentimentModel`

Tests the complete model with:
- GCN disabled (`use_dependency_gcn=False`) — returns logits directly from attention head
- GCN enabled with dummy dependency edges — triggers the GCN branch
- `return_attention=True` — returns a 4-tuple `(preds, attn, aspect_repr, gcn_out)`

In [ ]:
sep('Test 2: MultiAspectSentimentModel (GCN on/off + return_attention)')
B, S = 2, 64
ids  = torch.randint(0, 1000, (B, S))
mask = torch.ones(B, S, dtype=torch.long)
asp  = torch.zeros(B, dtype=torch.long)

# Test 2a: No GCN
try:
    cfg_no_gcn = {**config, 'model': {**config['model'], 'use_dependency_gcn': False}}
    m = create_model(cfg_no_gcn)
    m.eval()
    with torch.no_grad():
        out = m(ids, mask, asp)
    assert out.shape == (B, config['model']['num_classes'])
    ok(f'No-GCN: output shape {out.shape}')
except Exception as exc:
    err(f'No-GCN: {exc}')

# Test 2b: With GCN (dummy edges)
try:
    m = create_model(config)
    m.eval()
    edge_index = [
        torch.tensor([[0, 1, 2], [1, 2, 3]], dtype=torch.long),
        torch.tensor([[0, 1],    [1, 2]   ], dtype=torch.long),
    ]
    with torch.no_grad():
        out = m(ids, mask, asp, edge_index=edge_index)
    assert out.shape == (B, config['model']['num_classes'])
    ok(f'With-GCN: output shape {out.shape}')
except Exception as exc:
    err(f'With-GCN: {exc}')

# Test 2c: return_attention=True
try:
    m = create_model(config)
    m.eval()
    with torch.no_grad():
        out = m(ids, mask, asp, return_attention=True)
    assert isinstance(out, tuple) and len(out) == 4
    ok('return_attention=True returns 4-tuple (preds, attn, repr, gcn_out)')
except Exception as exc:
    err(f'return_attention: {exc}')

## Test 3 — Loss Functions

Verifies that all three loss classes produce a **non-NaN scalar** for random logits/labels:
- `FocalLoss` — focal weighting + class alpha weights
- `HybridLoss` — weighted combination of Focal + Class-Balanced + Dice
- `AspectSpecificLossManager` — manages one `HybridLoss` per aspect

In [ ]:
sep('Test 3: Loss Functions')
B = 8
logits = torch.randn(B, 3)
labels = torch.randint(0, 3, (B,))

# FocalLoss
try:
    fl   = FocalLoss(gamma=2.0, alpha=torch.tensor([1.0, 1.5, 1.2]))
    loss = fl(logits, labels)
    assert loss.shape == () and not torch.isnan(loss)
    ok(f'FocalLoss: {loss.item():.4f}')
except Exception as exc:
    err(f'FocalLoss: {exc}')

# HybridLoss
try:
    hl   = HybridLoss(samples_per_class=[100, 50, 200], focal_gamma=2.0, cb_beta=0.999,
                      weights={'focal': 1.0, 'cb': 0.5, 'dice': 0.3})
    loss = hl(logits, labels)
    assert not torch.isnan(loss)
    ok(f'HybridLoss: {loss.item():.4f}')
except Exception as exc:
    err(f'HybridLoss: {exc}')

# AspectSpecificLossManager
try:
    class_counts = config.get('training', {}).get('class_counts', {
        'stayingpower': [727, 244, 1076],
        'texture':      [639, 420, 2563],
        'smell':        [335, 274, 1668],
    })
    mgr          = AspectSpecificLossManager(class_counts, config['training'])
    aspect_ids   = torch.zeros(B, dtype=torch.long)
    aspects      = config['aspects']['names']
    loss, details = mgr.compute_loss(logits, labels, aspect_ids, aspects)
    assert not torch.isnan(loss)
    ok(f'AspectSpecificLossManager: {loss.item():.4f}')
except Exception as exc:
    err(f'AspectSpecificLossManager: {exc}')

## Test 4 — Ablation Flag Correctness

Verifies that every ablation configuration (A1–A5 variants) can run a forward pass
without crashing. None of the ablation flags should break the model's basic interface.

In [ ]:
sep('Test 4: Ablation flag combinations')
B, S = 2, 32
ids  = torch.randint(0, 1000, (B, S))
mask = torch.ones(B, S, dtype=torch.long)
asp  = torch.zeros(B, dtype=torch.long)

ablation_cases = [
    {'use_aspect_attention': False, 'use_dependency_gcn': True,  'use_shared_classifier': False},  # A2
    {'use_aspect_attention': True,  'use_dependency_gcn': False, 'use_shared_classifier': False},  # A1
    {'use_aspect_attention': True,  'use_dependency_gcn': True,  'use_shared_classifier': True},   # A5
    {'use_aspect_attention': False, 'use_dependency_gcn': False, 'use_shared_classifier': True},   # all off + shared
]
for flags in ablation_cases:
    label = '  '.join(f'{k}={v}' for k, v in flags.items())
    try:
        cfg = {**config, 'model': {**config['model'], **flags}}
        m   = create_model(cfg)
        m.eval()
        with torch.no_grad():
            out = m(ids, mask, asp)
        assert out.shape == (B, config['model']['num_classes'])
        ok(label)
    except Exception as exc:
        err(f'{label}: {exc}')

In [ ]:
sep('Test 5: Parameter count sanity (should be > 50 M for RoBERTa-base)')
try:
    m     = create_model(config)
    total = sum(p.numel() for p in m.parameters() if p.requires_grad)
    ok(f'Trainable parameters: {total:,}')
    assert total > 50_000_000, f'Suspiciously low: {total:,}'
    ok('Parameter count sanity check PASSED')
except Exception as exc:
    err(f'Parameter count: {exc}')

print()
print('='*60)
print('  All component tests complete.')
print('='*60)